# Analiza Opóźnień Komunikacji Miejskiej w Krakowie (GTFS-RT)

Niniejszy notatnik służy do analizy rzeczywistych opóźnień komunikacji miejskiej w Krakowie. Dane pobierane są w czasie rzeczywistym z usług [GTFS-RT ZTP Kraków](https://gtfs.ztp.krakow.pl/). 

Wykorzystujemy:
- **TripUpdates** (format `.pb` - Protobuf), aby pozyskać estymowane czasy przyjazdów i porównać je do planowanych.
- **Dane statyczne (GTFS)** do podpięcia lokalizacji geo (przystanków) oraz nazw linii.

Notatnik wygeneruje interaktywne mapy ulic ukazujące natężenie opóźnień, a także odpowiednie statystyki i wykresy.

In [ ]:
# Jeśli nie masz ich zainstalowanych, odkomentuj i uruchom poniższą linijkę:
# !pip install gtfs-realtime-bindings protobuf requests pandas plotly

import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def fetch_gtfs_rt_delays():
    print("Pobieranie aktualnych danych GTFS-RT (TripUpdates) dla Krakowa...")

    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb"
    }

    delays_data = []

    for v_type, url in urls.items():
        print(f" Pobieranie danych dla: {v_type}...")
        try:
            feed = gtfs_realtime_pb2.FeedMessage()
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            feed.ParseFromString(response.content)
            
            for entity in feed.entity:
                if entity.HasField('trip_update'):
                    trip_id = entity.trip_update.trip.trip_id
                    route_id = entity.trip_update.trip.route_id
                    
                    for stu in entity.trip_update.stop_time_update:
                        stop_id = stu.stop_id
                        delay = None
                        
                        # Pobieranie opóźnienia z przyjazdu lub odjazdu (preferujemy przyjazd)
                        if stu.HasField('arrival') and stu.arrival.HasField('delay'):
                            delay = stu.arrival.delay
                        elif stu.HasField('departure') and stu.departure.HasField('delay'):
                            delay = stu.departure.delay
                        
                        if delay is not None:
                            delays_data.append({
                                "typ": v_type,
                                "trip_id": trip_id,
                                "line_num": route_id,
                                "stop_id": stop_id,
                                "delay_sec": delay,
                                "delay_min": delay / 60.0
                            })
        except Exception as e:
            print(f"  Błąd podczas pobierania {v_type}: {e}")

    df_delays = pd.DataFrame(delays_data)
    
    if not df_delays.empty:
        # Skupmy się tylko na dodatnich opóźnieniach (przyjazdy po czasie)
        df_delays = df_delays[df_delays['delay_sec'] > 0]
        print(f"\nZakończono. Pobrano {len(df_delays)} rekordów z dodatnimi opóźnieniami.")
    else:
        print("\nNie udało się pobrać żadnych opóźnień lub brak opóźnień w tej chwili.")
        
    return df_delays

df_delays = fetch_gtfs_rt_delays()
df_delays.head()

In [ ]:
# Wczytanie fizycznych lokalizacji przystanków i nazw linii z rozkładów (GTFS Zip)
# Zakładamy, że historyczne (ale w miarę aktualne) pliki przystanków znajdują się lokalnie
static_gtfs_dir = "data/GTFS_ZTP_17.05.26/"
stops_file = os.path.join(static_gtfs_dir, "stops.txt")
routes_file = os.path.join(static_gtfs_dir, "routes.txt")

if os.path.exists(stops_file) and not df_delays.empty:
    df_stops = pd.read_csv(stops_file, dtype=str)
    # Konwersja coords na wartości numeryczne
    df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
    df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")
    
    # Łączenie przystanków
    df_merged = df_delays.merge(
        df_stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
        on='stop_id', 
        how='inner'
    )
    
    # Przypisywanie nazw linii, jeżeli istnieje routes.txt
    if os.path.exists(routes_file):
        df_routes = pd.read_csv(routes_file, dtype=str)
        df_merged = df_merged.merge(
            df_routes[['route_id', 'route_short_name']], 
            left_on='line_num', 
            right_on='route_id',
            how='left'
        )
        # Zastąpienie wewnętrznego ID linii jej nazwą publiczną np. "152"
        df_merged['linia'] = df_merged['route_short_name'].fillna(df_merged['line_num'])
    else:
        df_merged['linia'] = df_merged['line_num']

    # Obliczanie średniego opóźnienia, maksymalnego, oraz ilości pojazdów per Przystanek
    df_stops_delays = df_merged.groupby(['stop_name', 'stop_lat', 'stop_lon', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        max_delay_min=('delay_min', 'max'),
        measurements_count=('delay_min', 'count')
    )
    
    # Wyświetlamy tylko te przystanki, przez które opóźnione przejeżdża więcej niż x pojazdów
    df_stops_delays = df_stops_delays[df_stops_delays['measurements_count'] >= 2]
    
    display(df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head())
else:
    print("Brak pliku przystanków lub brak punktów pobranych - upewnij się, ze ścieżka do stops.txt jest poprawna.")

## Mapa Opóźnień (Heatmap / Punkty uliczne)
Poniżej wygenerowana interaktywna mapa prezentująca przystanki ze średnimi opóźnieniami na krakowskich ulicach. Korzystamy z siatki ulicy OpenStreetMap, gdzie zabarwienie punktu odpowiada długości opóźnienia.

In [ ]:
if 'df_stops_delays' in locals() and not df_stops_delays.empty:
    fig_map = px.scatter_mapbox(
        df_stops_delays,
        lat="stop_lat",
        lon="stop_lon",
        color="mean_delay_min",
        size="mean_delay_min",
        hover_name="stop_name",
        hover_data={
            "mean_delay_min": ":.1f", 
            "max_delay_min": ":.1f", 
            "measurements_count": True, 
            "typ": True, 
            "stop_lat": False, 
            "stop_lon": False
        },
        color_continuous_scale=px.colors.sequential.Inferno,
        zoom=11.5,
        center={"lat": 50.0614, "lon": 19.9383}, # Śródmieście Krakowa
        title="Aktualne średnie opóźnienia komunikacji miejskiej (w minutach)",
        height=700
    )

    fig_map.update_layout(
        mapbox_style="open-street-map",
        margin={"r":0,"t":40,"l":0,"b":0}
    )
    fig_map.show()
else:
    print("Nie ma danych do wykreślenia mapy.")

In [ ]:
if 'df_stops_delays' in locals() and not df_stops_delays.empty:
    fig_density = px.density_mapbox(
        df_stops_delays,
        lat='stop_lat',
        lon='stop_lon',
        z='mean_delay_min',
        radius=12,
        center=dict(lat=50.0614, lon=19.9383),
        zoom=11.5,
        mapbox_style="open-street-map",
        title="Mapa zagęszczenia (Heatmap) największych opóźnień komunikacji w Krakowie",
        height=700
    )
    
    fig_density.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig_density.show()

## Wykresy (Gdzie są największe opóźnienia)
Przeanalizujmy, które przystanki oraz które linie notują średnio największe opóźnienia w pozyskanej próbce czasowej.

In [ ]:
if 'df_stops_delays' in locals() and not df_stops_delays.empty:
    # Top 15 Przystanków
    top_15_mean = df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head(15)

    fig_bar_stops = px.bar(
        top_15_mean,
        x='stop_name',
        y='mean_delay_min',
        color='typ',
        title="Top 15 przystanków o największym średnim opóźnieniu",
        labels={'stop_name': 'Przystanek', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_stops.update_layout(xaxis_tickangle=-45)
    fig_bar_stops.show()
    
if 'df_merged' in locals() and not df_merged.empty:
    # Agregacja po Liniach
    df_route_delays = df_merged.groupby(['linia', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        measurements_count=('delay_min', 'count')
    )
    
    # Filtrujemy by odrzucić pojedyncze strzały pomiarów dla jakiejś trasy
    df_route_delays = df_route_delays[df_route_delays['measurements_count'] >= 3]
    
    top_15_routes = df_route_delays.sort_values(by='mean_delay_min', ascending=False).head(15)
    
    fig_bar_routes = px.bar(
        top_15_routes,
        x='linia',
        y='mean_delay_min',
        color='typ',
        title="Top 15 linii komunikacyjnych o największym średnim opóźnieniu",
        labels={'linia': 'Numer Linii', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_routes.update_layout(xaxis_type='category') # By numery linii zachowywały się jak kategorie
    fig_bar_routes.show()